In [92]:
%cd /content
!rm -rf gen_aps
!git clone https://github.com/maxzawalo/gen_aps.git

/content
Cloning into 'gen_aps'...
remote: Enumerating objects: 14, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 14 (delta 5), reused 11 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (14/14), 382.09 KiB | 17.37 MiB/s, done.
Resolving deltas: 100% (5/5), done.


In [93]:
%cd gen_aps

/content/gen_aps


In [94]:
%%writefile task.md

# Строка после решетки (#) игнорируется (комментарий)
[СМЕНЫ]
# Название		| Рабочее время
Выходной		| 09:00-23:00
Выходной		| 23:00-00:00
Стиралка		| 07:00-19:00

[РАСПИСАНИЕ]
# ИмяРесурса	| Дата			| Смена			| Приоритет
Я				| 04.07.2026	| Выходной		| 10
Чайник			| 04.07.2026	| Выходной		| 10
Стиралка		| 04.07.2026	| Стиралка		| 10
Комп			| 04.07.2026	| Выходной		| 10

[ЗАДАЧИ]
# Название			| Длительность	| Ресурсы (через запятую)
Завтрак				| 15 мин		| Я
Смеситель			| 30 мин		| Я
Посмотреть фильм	| 1.5 часа		| Я, Комп
Выгрузить белье		| 5 мин			| Я, Стиралка
Уборка            | 1ч	  		| Я

[ЗАДАЧИ С ТРИГГЕРОМ]
#					| 			Триггер			| 			Задача
# Название			| Длительность	| Ресурсы	| Длительность	| Ресурсы
Скачать фильм		| 	1 мин		| Я, Комп	|	20 мин		| Комп
Стирка				| 	1 мин		| Я, Стиралка|	60 мин		| Стиралка
Вскипятить чайник	| 	1 мин		| Я, Чайник	|	5 мин		| Чайник

[СРАЗУ ПОСЛЕ]
# Символ разделитель (->) лучше копипастить

[ПОСЛЕДОВАТЕЛЬНО]
# Символ разделитель (->) лучше копипастить
Скачать фильм -> Посмотреть фильм
Вскипятить чайник -> Завтрак
Завтрак -> Смеситель
Смеситель -> Уборка

[ЦЕПОЧКИ]
# Не пересекаются друг с другом. Одиночные задачи могут "вклиниваться".
# Цепочка					| Кол-во повторов
Стирка -> Выгрузить белье	| 2				# В цепочку автоматически добавляется Триггер

Overwriting task.md


In [98]:
!java -Xlog:disable -jar mcts_aps.jar task.md

📖 Конфигурация успешно загружена!
🗓️ ГЕНЕРАЦИЯ ОПТИМАЛЬНОГО РАСПИСАНИЯ ЧЕРЕЗ MCTS...

=== ИТОГОВОЕ ПОМИНУТНОЕ РАСПИСАНИЕ ===
[09:00 - 09:01] -> Скачать фильм (Триггер) (Ресурсы: [Я, Комп])
[09:01 - 09:21] -> Скачать фильм в фоне (Ресурсы: [Комп])
[09:01 - 09:02] -> Вскипятить чайник (Триггер) (Ресурсы: [Я, Чайник])
[09:02 - 09:07] -> Вскипятить чайник в фоне (Ресурсы: [Чайник])
[09:02 - 09:03] -> Стирка (Триггер) #1 (Ресурсы: [Я, Стиралка])
[09:03 - 10:03] -> Стирка в фоне #1 (Ресурсы: [Стиралка])
[09:07 - 09:22] -> Завтрак (Ресурсы: [Я])
[09:22 - 09:52] -> Смеситель (Ресурсы: [Я])
[09:52 - 10:52] -> Уборка (Ресурсы: [Я])
[10:52 - 10:57] -> Выгрузить белье #1 (Ресурсы: [Я, Стиралка])
[10:57 - 10:58] -> Стирка (Триггер) #2 (Ресурсы: [Я, Стиралка])
[10:58 - 11:58] -> Стирка в фоне #2 (Ресурсы: [Стиралка])
[10:58 - 12:28] -> Посмотреть фильм (Ресурсы: [Я, Комп])
[12:28 - 12:33] -> Выгрузить белье #2 (Ресурсы: [Я, Стиралка])
💾 Файл Vis-Timeline успешно обновлен: data/plan.html


### Очистка html

In [99]:
from bs4 import BeautifulSoup

# Читаем файл
with open('data/plan.html', 'r', encoding='utf-8') as f:
    html_content = f.read()

soup = BeautifulSoup(html_content, 'html.parser')

# Находим и удаляем скрипт с локальным путем
local_script = soup.find('script', src="vis-timeline-graph2d.min.js")
if local_script:
    local_script.decompose()

# Находим и удаляем стили с локальным путем
local_link = soup.find('link', href="vis-timeline-graph2d.min.css")
if local_link:
    local_link.decompose()

# Сохраняем чистый файл обратно (formatter=None сохранит оригинальные теги)
with open('data/plan.html', 'w', encoding='utf-8') as f:
    f.write(str(soup))

### Показываем план

In [100]:
import IPython

# Читаем сгенерированный вашей программой файл
with open('data/plan.html', 'r', encoding='utf-8') as f:
    html_content = f.read()

# Безопасный вывод диаграммы в Colab
IPython.display.display(IPython.display.HTML(
    f'''
    <iframe srcdoc="{html_content.replace('"', '&quot;')}"
            style="width: 100%; height: 500px; border: none;"
            sandbox="allow-scripts allow-popups allow-forms allow-modals">
    </iframe>
    '''
))
